# Phase 2 — Step 4: improving A2 (causal backbone + right context, less memorising)

**What we learned in Step 3** (A2, full fine-tuning, fold-2 macro-F1):

| attention                  | macro-F1  |
| -------------------------- | --------- |
| bidirectional (switched)   | 0.419     |
| **causal (as pretrained)** | **0.497** |

A2 learned to read left→right in Phase 1; 35 documents are not enough to re-learn reading both ways.
But a causal model cannot see the RIGHT side of a word (at "Kolkata" it has not yet read "Municipal Corporation"),
and the model memorised its 223 training windows (training loss → 0.002).

**This notebook tries three fixes, each alone (Stage A only, ~3 min each), then compares:**

| run      | idea                                                                                                                                      |
| -------- | ----------------------------------------------------------------------------------------------------------------------------------------- |
| `look4`  | **lookahead head** (our idea): tag token _t_ using the causal vectors of tokens _t … t+4_ → 4 tokens of right context, backbone untouched |
| `bilstm` | **BiLSTM head**: a small bidirectional LSTM on top of the causal vectors → full right context                                             |
| `aug`    | **random window offsets**: every epoch the documents are cut at different places (data augmentation against memorising)                   |

All code is in `src/ner_experiment.py` (one config → train, report, submit), shared by all later experiments.
Everything is cached: re-running skips finished runs.


## 0. Setup


In [1]:
import sys, importlib
from pathlib import Path
import torch

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".gitignore").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Project root with .gitignore not found")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import ner_utils as nu, ner_models as nm, ner_experiment as ne
for m in (nu, nm, ne):
    importlib.reload(m)

c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. The sweep (Stage A for each setting: train fold 1, validate fold 2)

The first entry is the Step-3 causal run (already finished → loaded from cache).


In [2]:
SWEEP = [
    {},                                        # causal + linear head (Step 3, cached)
    {"head": "lookahead", "lookahead": 4},     # + right context through the head
    {"head": "bilstm"},                        # + full right context through a BiLSTM
    {"augment": True},                         # + random window offsets
]
for overrides in SWEEP:
    exp = ne.Experiment(PROJECT_ROOT, overrides)
    exp.stage_a()
    exp.report(n_examples=0)
    exp.log()
    del exp
    torch.cuda.empty_cache()
    print("=" * 100)

experiment: A2_full_causal_norm_hm0 | device cuda | precision torch.bfloat16
parameters: 70.07M total, 70.07M trainable | loaded {'arch': 'A2', 'layers': 10, 'loaded_tensors': 72, 'dropped_tensors': 5, 'dropped_groups': ['copy_gate', 'copy_k', 'copy_q', 'lm_head']}
[cache] training 'fold1' already done -> loaded best_fold1.pt (best epoch 4)
Stage A: best epoch 4 | val macro-F1 0.4975 | 0s
A2_full_causal_norm_hm0 - fold 2 (validation):
  macro-F1 0.4975 | macro-P 0.5176 | macro-R 0.4992 | F1(of macro P,R) 0.5082 | micro-F1 0.5934
  label           P      R     F1    TP    FP    FN  gold
  APP         0.218  0.290  0.249    58   208   142   200
  RESP        0.316  0.163  0.215    31    67   159   190
  A.COUNSEL   0.485  0.504  0.494    63    67    62   125
  R.COUNSEL   0.500  0.505  0.503    49    49    48    97
  JUDGE       0.612  0.697  0.651    85    54    37   122
  WIT         0.432  0.158  0.232    19    25   101   120
  AUTH        0.503  0.316  0.388   148   146   320   468
 

epoch  1 | train loss 1.2088 | 13s | val macro-F1 0.2526 micro-F1 0.3951  <- best


epoch  2 | train loss 0.1289 | 12s | val macro-F1 0.2561 micro-F1 0.3988  <- best


epoch  3 | train loss 0.0515 | 13s | val macro-F1 0.3932 micro-F1 0.5333  <- best


epoch  4 | train loss 0.0275 | 13s | val macro-F1 0.4369 micro-F1 0.5557  <- best


epoch  5 | train loss 0.0186 | 13s | val macro-F1 0.4297 micro-F1 0.5715


... (35 outputs left out)

epoch 10 | train loss 0.0027 | 12s | val macro-F1 0.4274 micro-F1 0.5651


epoch 11 | train loss 0.0016 | 12s | val macro-F1 0.4439 micro-F1 0.5728


epoch 12 | train loss 0.0024 | 11s | val macro-F1 0.4337 micro-F1 0.5597
early stop: no improvement for 4 epochs
training 'fold1' finished -> loaded best epoch 8
Stage A: best epoch 8 | val macro-F1 0.4485 | 220s


A2_full_causal_norm_hm0_aug - fold 2 (validation):
  macro-F1 0.4485 | macro-P 0.4515 | macro-R 0.4713 | F1(of macro P,R) 0.4612 | micro-F1 0.5639
  label           P      R     F1    TP    FP    FN  gold
  APP         0.129  0.135  0.132    27   182   173   200
  RESP        0.068  0.021  0.032     4    55   186   190
  A.COUNSEL   0.381  0.600  0.466    75   122    50   125
  R.COUNSEL   0.531  0.526  0.528    51    45    46    97
  JUDGE       0.455  0.533  0.491    65    78    57   122
  WIT         0.455  0.167  0.244    20    24   100   120
  AUTH        0.379  0.231  0.287   108   177   360   468
  COURT       0.760  0.799  0.779   446   141   112   558
  STAT        0.614  0.668  0.640   491   309   244   735
  PREC        0.357  0.492  0.414   275   495   284   559
  DATE        0.929  0.948  0.938   722    55    40   762
  CASENO      0.361  0.537  0.431   181   321   156   337
decoding on the same outputs: {'argmax_trim0': 0.34, 'argmax_trim1': 0.3433, 'viterbi_trim0': 0.444

## 2. Comparison (all finished A2 runs, sorted by fold-2 macro-F1)


In [3]:
rows = ne.compare(PROJECT_ROOT)

run                                         macro  micro  ep     APP   RESP A.COUN R.COUN  JUDGE    WIT   AUTH  COURT   STAT   PREC   DATE CASENO
A2_full_causal_bilstm_norm_hm0             0.5057 0.6133   9   0.208  0.185  0.454  0.505  0.690  0.245  0.361  0.814  0.708  0.476  0.948  0.473
A2_full_causal_norm_hm0                    0.4975 0.5934   4   0.249  0.215  0.494  0.503  0.651  0.232  0.389  0.784  0.676  0.429  0.941  0.407
A2_full_causal_look4_norm_hm0              0.4698 0.5928  12   0.141  0.162  0.335  0.464  0.591  0.245  0.373  0.810  0.672  0.383  0.940  0.523
A2_full_causal_norm_hm0_aug                0.4485 0.5639   8   0.132  0.032  0.466  0.528  0.491  0.244  0.287  0.779  0.640  0.414  0.938  0.431
A2_full_bidir_norm_hm0                     0.4190 0.5523   8   0.107  0.091  0.310  0.318  0.519  0.226  0.269  0.785  0.641  0.389  0.927  0.446


## What to tell me

Just say "done" — I read the results from the run folders. (Or paste the table from cell 2.)
